In [ ]:
# jaccard like
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

base_dir = "/path/to/project"


cha_pan = os.path.join(base_dir, "final_analysis/data/CHA/pan/bp35w60")
cha_ngs = os.path.join(base_dir, "final_analysis/data/CHA/ngs/bp35w60")
chb_unmasked_dir = os.path.join(base_dir, "map_chm13paper/unmask/CHB")
chs_unmasked_dir = os.path.join(base_dir, "map_chm13paper/unmask/CHS")
jpt_unmasked_dir = os.path.join(base_dir, "map_chm13paper/unmask/JPT")
khv_unmasked_dir = os.path.join(base_dir, "map_chm13paper/unmask/KHV")
cdx_unmasked_dir = os.path.join(base_dir, "map_chm13paper/unmask/CDX")
fin_unmasked_dir = os.path.join(base_dir, "map_chm13paper/unmask/FIN")
ceu_unmasked_dir = os.path.join(base_dir, "map_chm13paper/unmask/CEU")
yri_unmasked_dir = os.path.join(base_dir, "map_chm13paper/unmask/YRI")


datasets = {
    "CHA_pan": cha_pan,
    "CHA_ngs": cha_ngs,
    "CHB": chb_unmasked_dir,
    "CHS": chs_unmasked_dir,
    "JPT": jpt_unmasked_dir,
    "KHV": khv_unmasked_dir,
    "CDX": cdx_unmasked_dir,
    "FIN": fin_unmasked_dir,
    "CEU": ceu_unmasked_dir,
    "YRI": yri_unmasked_dir,
}



def load_map(directory, chrom):
    if directory == chs_unmasked_dir:
        chs_path = os.path.join(chs_unmasked_dir, f"CHS_chr{chrom}_no_mask.txt")
        dataframe = pd.read_csv(chs_path)
    elif directory == cha_pan:
        cha_pan_path = os.path.join(cha_pan, f"CHA_recombmap_chr{chrom}_bp35w60")
        dataframe = pd.read_csv(cha_pan_path, sep="\t", header=None, names=["Start", "End", "Rec.Rate"])
    elif directory == cha_ngs:
        cha_ngs_path = os.path.join(cha_ngs, f"CHA_recombmap_chr{chrom}_bp35w60")
        dataframe = pd.read_csv(cha_ngs_path, sep="\t", header=None, names=["Start", "End", "Rec.Rate"])
    elif directory == chb_unmasked_dir:
        chb_path = os.path.join(chb_unmasked_dir, f"CHB_chr{chrom}_no_mask.txt")
        dataframe = pd.read_csv(chb_path)
    elif directory == jpt_unmasked_dir:
        jpt_path = os.path.join(jpt_unmasked_dir, f"JPT_chr{chrom}_no_mask.txt")
        dataframe = pd.read_csv(jpt_path)
    elif directory == khv_unmasked_dir:
        khv_path = os.path.join(khv_unmasked_dir, f"KHV_chr{chrom}_no_mask.txt")
        dataframe = pd.read_csv(khv_path)
    elif directory == cdx_unmasked_dir:
        cdx_path = os.path.join(cdx_unmasked_dir, f"CDX_chr{chrom}_no_mask.txt")
        dataframe = pd.read_csv(cdx_path)
    elif directory == fin_unmasked_dir:
        fin_path = os.path.join(fin_unmasked_dir, f"FIN_chr{chrom}_no_mask.txt")
        dataframe = pd.read_csv(fin_path)
    elif directory == yri_unmasked_dir:
        yri_path = os.path.join(yri_unmasked_dir, f"YRI_chr{chrom}_no_mask.txt")
        dataframe = pd.read_csv(yri_path)
    elif directory == ceu_unmasked_dir:
        ceu_path = os.path.join(ceu_unmasked_dir, f"CEU_chr{chrom}_no_mask.txt")
        dataframe = pd.read_csv(ceu_path)
    else:
        raise ValueError(f"Unknown directory: {directory}")
    return dataframe



def filter_outliers(df):
    return df[df["Rec.Rate"] <= 1e-5]


def weighted_mean(df):
    lengths = df["End"] - df["Start"]
    return np.average(df["Rec.Rate"], weights=lengths)



def extract_hotspots(df, mean_rate):
    threshold = 10 * mean_rate
    return df[df["Rec.Rate"] >= threshold]





In [ ]:
def clip_map_to_allowed_regions(df_map: pd.DataFrame,
                               allowed_df: pd.DataFrame,
                               chrom: str,
                               chrom_col_allowed: str = "chr",
                               start_col_allowed: str = "Start",
                               end_col_allowed: str = "End") -> pd.DataFrame:

    # Allowed intervals for this chromosome
    allowed = allowed_df[allowed_df[chrom_col_allowed].astype(str) == str(chrom)][
        [start_col_allowed, end_col_allowed]
    ].copy()

    if allowed.empty or df_map.empty:
        return df_map.iloc[0:0].copy()

    # sort for safety
    allowed = allowed.sort_values([start_col_allowed, end_col_allowed]).to_numpy()
    df_map = df_map.sort_values(["Start", "End"]).reset_index(drop=True)

    out_rows = []

    j = 0
    for s, e, r in df_map[["Start", "End", "Rec.Rate"]].to_numpy():
        if e <= s:
            continue

        # advance allowed pointer until it might overlap
        while j < len(allowed) and allowed[j][1] <= s:
            j += 1

        k = j
        # collect all overlaps with allowed intervals
        while k < len(allowed) and allowed[k][0] < e:
            a_s, a_e = allowed[k]
            ov_s = max(s, a_s)
            ov_e = min(e, a_e)
            if ov_s < ov_e:
                out_rows.append((ov_s, ov_e, r))
            if a_e >= e:
                break
            k += 1

    if not out_rows:
        return df_map.iloc[0:0].copy()

    df_out = pd.DataFrame(out_rows, columns=["Start", "End", "Rec.Rate"])
    df_out = df_out.sort_values(["Start", "End"]).reset_index(drop=True)
    return df_out





In [ ]:
def make_1kb_windows(allowed_df, window=1000):
    out = []

    for _, r in allowed_df.iterrows():
        chrom = r["chr"]
        for s in range(int(r["Start"]), int(r["End"]), window):
            e = s + window
            if e <= r["End"]:  # keep full windows only
                out.append((chrom, s, e))

    return pd.DataFrame(out, columns=["chrom", "Start", "End"])


In [ ]:
def extract_hotspot_windows(df_map, windows_chr, fold=10):
    """
    Returns BED-like dataframe of hotspot windows
    """
    mean_rate = weighted_mean(df_map)
    threshold = fold * mean_rate

    hotspot_rows = []

    for _, w in windows_chr.iterrows():
        ov = df_map[
            (df_map["Start"] < w["End"]) &
            (df_map["End"] > w["Start"]) &
            (df_map["Rec.Rate"] >= threshold)
        ]
        if len(ov) > 0:
            hotspot_rows.append(w)

    if len(hotspot_rows) == 0:
        return pd.DataFrame(columns=["chrom", "Start", "End"])

    return pd.DataFrame(hotspot_rows)


In [ ]:
all_hotspots = {name: [] for name in datasets}
chromosomes = list(range(1, 23))


pan_available = os.path.join(base_dir, "final_analysis/data/cha_pan_availableregion/CHM13v2.telo_cent.complement.bed")
pan_available_region = pd.read_csv(
	pan_available, sep="\t", header=None, names=["Chrom", "Start", "End"]
)
# remove chrX and chrY in Chrom
pan_available_region = pan_available_region[~pan_available_region["Chrom"].isin(["chrX", "chrY"])]
pan_available_region ["chr"] = pan_available_region ["Chrom"].str.replace("chr", "")
pan_available_region


windows_1kb = make_1kb_windows(pan_available_region)
windows_1kb["chrom"] = windows_1kb["chrom"].astype(str)



In [ ]:
for chrom in chromosomes:
    print(f"Chromosome {chrom}")
    chrom = str(chrom)

    windows_chr = windows_1kb[windows_1kb["chrom"] == chrom]

    for name, directory in datasets.items():
        df = load_map(directory, chrom)

        df = filter_outliers(df)
        df = clip_map_to_allowed_regions(
            df_map=df,
            allowed_df=pan_available_region,
            chrom=str(chrom),
            chrom_col_allowed="chr",
            start_col_allowed="Start",
            end_col_allowed="End",
        )

        df = df.assign(chrom=chrom)

        # NEW hotspot definition
        hotspots = extract_hotspot_windows(df, windows_chr)
        print(len(hotspots))
        all_hotspots[name].append(hotspots)

for name in all_hotspots:
    all_hotspots[name] = pd.concat(all_hotspots[name], ignore_index=True)
    all_hotspots[name] = all_hotspots[name][["chrom", "Start", "End"]]



names = list(all_hotspots.keys())
N = len(names)

# Give each window a unique ID
for name in names:
    df = all_hotspots[name]
    all_hotspots[name] = df.assign(
        win_id=df["chrom"].astype(str)
        + ":" + df["Start"].astype(str)
        + "-" + df["End"].astype(str)
    )

# All windows observed in any population
all_windows = sorted(
    set().union(*[set(all_hotspots[name]["win_id"]) for name in names])
)

win_index = {w: i for i, w in enumerate(all_windows)}
pop_index = {p: i for i, p in enumerate(names)}

# Boolean window × population matrix
M = np.zeros((len(all_windows), N), dtype=bool)

for pop in names:
    j = pop_index[pop]
    for w in all_hotspots[pop]["win_id"]:
        M[win_index[w], j] = True



percent_matrix = np.full((N, N), np.nan)

# |A| for each population
counts = M.sum(axis=0)
print(counts)

for i in range(N):
    for j in range(N):
        if i == j:
            continue

        # |A ∩ B|
        intersection = np.logical_and(M[:, i], M[:, j]).sum()

        
        pct_shared = (2 * intersection) / (counts[i] + counts[j])
        percent_matrix[i, j] = pct_shared * 100
        print(percent_matrix[i, j])

print(percent_matrix)



In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

cmap = mcolors.LinearSegmentedColormap.from_list(
    "white_red",
    ["white", "red"]
)

# Make a copy of the colormap so you don't affect global state

# Set NaN ("bad") color to black
cmap.set_bad(color="#4d4d4d")

plt.figure(figsize=(10, 8))
plt.imshow(percent_matrix, cmap=cmap, interpolation="nearest")
plt.colorbar(label="Percentage of overlapping hotspots (%)")

plt.xticks(range(N), names, rotation=45, ha="right")
plt.yticks(range(N), names)
plt.tight_layout()
plt.show()
plt.savefig("hotspot_overlap.svg")